# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Muhammad-Ahmed-Zia/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one content_id, summarized over month=2026-03 (a mid-panel
month, deliberately not the sealed _sample/final month, since that
would let the future leak into any before/after label logic).

Underlying grain: at the raw fact-table level, one row = one
(client_hash_id, content_hash_id, report_date) — a single page's daily
performance snapshot. I aggregate this up to one row per content_id
for my lane's actual unit of analysis (Lane 2: Refresh / Content
Opportunity Scoring).

Tables used: fact_content_daily_performance (daily fact table,
filtered to month=2026-03) joined to dim_content (for content_age via
content_created_at).

In [6]:
%pip install -q duckdb huggingface_hub

import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "2026-03"

from huggingface_hub import HfApi
api = HfApi()
files = api.list_repo_files("FlyRank/internship-warehouse", repo_type="dataset")
fact_files = [f for f in files if "fact_content_daily_performance" in f and MONTH in f]
content_files = [f for f in files if "dim_content" in f]
print("Fact files for", MONTH, ":", fact_files[:5])
print("dim_content files:", content_files[:5])

Fact files for 2026-03 : ['fact_content_daily_performance/month=2026-03/data_0.parquet']
dim_content files: ['dim_content.parquet']


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Label/proxy: whether a page's visibility declined within the window —
average daily impressions in the first half of the month vs the
second half; second half meaningfully lower = is_declining.

Features (all knowable at the decision moment):
- avg_daily_impressions — GSC logs impressions as they happen.
- avg_position — observed, already-happened Search Console measurement.
- days_with_impressions — pure count of observed visibility days.
- content_age_days — from content_created_date, fixed at publish time.
- avg_impr_first_half — built only from the first half of the SAME
  window, never reaching past the decision point.

Context (not a feature, just for interpretation): content_hash_id,
client_hash_id — join keys only, never fed to a model.

Excluded, with why: FlyRank's product decision flags (health_score,
priority_score, action_type) — not shipped on purpose, to avoid the
circular-result trap. Below I deliberately break this rule once, on
purpose, to prove why it matters, then remove it.

In [11]:

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

features_df = con.sql(f"""
    WITH daily AS (
        SELECT * FROM read_parquet('{REL}/**/fact_content_daily_performance/**/month={MONTH}/*.parquet')
    ),
    halves AS (
        SELECT
            content_hash_id,
            AVG(gsc_impressions) FILTER (WHERE report_date < DATE '{MONTH}-16') AS avg_impr_first_half,
            AVG(gsc_impressions) FILTER (WHERE report_date >= DATE '{MONTH}-16') AS avg_impr_second_half,
            AVG(gsc_impressions) AS avg_daily_impressions,
            AVG(gsc_avg_position) AS avg_position,
            COUNT(*) FILTER (WHERE gsc_impressions > 0) AS days_with_impressions
        FROM daily
        GROUP BY content_hash_id
    )
    SELECT h.*, c.content_created_date
    FROM halves h
    LEFT JOIN read_parquet('{REL}/dim_content.parquet') c USING (content_hash_id)
""").df()

features_df["content_age_days"] = (
    pd.Timestamp(f"{MONTH}-01") - pd.to_datetime(features_df["content_created_date"])
).dt.days
features_df = features_df.dropna(subset=["avg_impr_first_half", "avg_impr_second_half"])
features_df["is_declining"] = (
    features_df["avg_impr_second_half"] < features_df["avg_impr_first_half"]
).astype(int)

feature_cols = ["avg_daily_impressions", "avg_position", "days_with_impressions",
                 "content_age_days", "avg_impr_first_half"]
print(features_df[feature_cols + ["is_declining"]].describe())

# Honest model
X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(
    features_df[feature_cols].fillna(0), features_df["is_declining"], test_size=0.25, random_state=42)
model = LogisticRegression(max_iter=1000).fit(X_train_h, y_train_h)
print("Honest score:", model.score(X_test_h, y_test_h))

# THE TRAP
features_df["pct_change_full_month"] = (
    (features_df["avg_impr_second_half"] - features_df["avg_impr_first_half"])
    / features_df["avg_impr_first_half"].replace(0, 1)
)
X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(
    features_df[feature_cols + ["pct_change_full_month"]].fillna(0), features_df["is_declining"], test_size=0.25, random_state=42)
leaky_model = LogisticRegression(max_iter=1000).fit(X_train_l, y_train_l)
print("Leaky score:", leaky_model.score(X_test_l, y_test_l), "-> jumps toward perfect, it's the label in disguise")




FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

       avg_daily_impressions   avg_position  days_with_impressions  \
count          319758.000000  174233.000000          319758.000000   
mean               28.452030      16.023329              11.254289   
std               132.912593      17.701282              13.291775   
min                 0.000000       0.000000               0.000000   
25%                 0.000000       5.011775               0.000000   
50%                 0.068966       8.531185               2.000000   
75%                 7.838710      20.406645              29.000000   
max             19907.225806     309.000000              31.000000   

       content_age_days  avg_impr_first_half   is_declining  
count     319758.000000        319758.000000  319758.000000  
mean         185.124141            26.625425       0.236691  
std          116.339984           126.163034       0.425052  
min          -19.000000             0.000000       0.000000  
25%           80.000000             0.000000       0.000000

In [8]:
features_df = features_df.drop(columns=["pct_change_full_month"])
print("Leak removed. Honest score stands:", model.score(X_test_h, y_test_h))

Leak removed. Honest score stands: 0.9796597448086064


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Three claims from Section 1, each proven with a query below: the
grain is really (client_hash_id, content_hash_id, report_date); the slice's row
count and date span match month=2026-03; and availability, filtered
with IS TRUE, shows how many rows actually have usable data.

In [9]:
# Section 3: the three verification queries
grain_check = con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           COUNT(DISTINCT client_hash_id || '|' || content_hash_id || '|' || report_date::VARCHAR) AS distinct_keys
    FROM read_parquet('{REL}/**/fact_content_daily_performance/**/month={MONTH}/*.parquet')
""").df()
print(grain_check)
print("Grain confirmed:", grain_check["total_rows"][0] == grain_check["distinct_keys"][0])

span_check = con.sql(f"""
    SELECT COUNT(*) AS row_count, MIN(report_date) AS min_date, MAX(report_date) AS max_date
    FROM read_parquet('{REL}/**/fact_content_daily_performance/**/month={MONTH}/*.parquet')
""").df()
print(span_check)

availability_check = con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS rows_with_ga4,
           COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) * 1.0 / COUNT(*) AS pct_with_ga4
    FROM read_parquet('{REL}/**/fact_content_daily_performance/**/month={MONTH}/*.parquet')
""").df()
print(availability_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  distinct_keys
0     9841378        9841378
Grain confirmed: True
   row_count   min_date   max_date
0    9841378 2026-03-01 2026-03-31
   total_rows  rows_with_ga4  pct_with_ga4
0     9841378         413966      0.042064


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This is an unbalanced panel — different clients have different history
depths, and only 9 of 70 clients have 12+ months of history. GA4 coverage in this slice is sparse (4.2%, 413,966 of 9,841,378 rows),
so most rows are GSC-only — GA4-derived features would have far less
support than GSC-derived ones. A small number of rows show negative content_age_days (min -19),
likely late-arriving metadata rather than true future-dated content. A single
mid-panel month may underrepresent clients who onboarded later in the
year. A first-half-vs-second-half comparison within one month also
can't distinguish a real sustained decline from ordinary week-to-week
noise, a mid-month reporting gap, or seasonality/consolidation (per
the lane guide Section 7) — it would need a longer window plus a
persistence check to call something a genuine decline rather than noise.

In [10]:
# Section 4: client coverage check — dim_clients is likely flat too
client_coverage = con.sql(f"""
    SELECT COUNT(DISTINCT client_hash_id) AS clients_in_slice
    FROM read_parquet('{REL}/**/fact_content_daily_performance/**/month={MONTH}/*.parquet')
""").df()
print(client_coverage)



   clients_in_slice
0                55


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.